# time-stage-instrumentation — ex2: context-manager `stage` helper that accumulates per-name elapsed time across nested calls

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `time-stage-instrumentation`. Running the final beacon cell reports progress against the `Logging: time-stage instrumentation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: time-stage instrumentation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`time-stage-instrumentation`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "time-stage-instrumentation"
DD_SUBTOPIC = "Logging: time-stage instrumentation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Per-stage context manager + nested re-entry

Ex1 used inline `t0 = perf_counter()` ... `acc += perf_counter() - t0`. The deepening move is a context-manager helper:

```python
@contextmanager
def stage(name, acc):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        acc[name] = acc.get(name, 0.0) + (time.perf_counter() - t0)
```

**Why `finally`.** A raised exception inside the block still records the partial elapsed time. Without `finally`, an OOM mid-forward leaves the accumulator silently missing that iteration's contribution.

**Re-entrant safety.** Calling `stage('forward', acc)` twice in nested scopes accumulates BOTH elapsed times into the same key — total is the sum of (outer block + inner block). For non-overlapping stages this is what you want; for overlapping stages you need per-frame keys.

### Exercise 2 — context-manager `stage` helper that accumulates per-name elapsed time across nested calls

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `@contextmanager` + `try/finally` over `time.perf_counter()` to build a `stage(name, acc)` helper that accumulates per-stage seconds even when the wrapped block raises, and supports nested (non-overlapping) re-entry into the same accumulator.
> Keywords: context-manager, perf_counter, accumulate, exception-safe
> ```

**KCs targeted:** `contextmanager-finally-accumulate`, `dict-get-default-accumulator`

Implement `ex2_stage(name, acc)`. A reusable context manager for per-stage timing with crash-safe accumulation.

Contract:

1. Decorated with `@contextlib.contextmanager`.
2. Records `t0 = time.perf_counter()` at entry.
3. `yield` (no value needed).
4. In a `finally` block: compute `elapsed = time.perf_counter() - t0` and do `acc[name] = acc.get(name, 0.0) + elapsed`. Use `.get(name, 0.0)` so a fresh accumulator dict works.
5. If an exception is raised inside the block, the `finally` still records the elapsed time, then the exception propagates naturally (do NOT swallow it).

Inputs:
- `name`: `str`, key into `acc`.
- `acc`: `dict[str, float]`, mutable accumulator.

Output: context manager (you don't return a value from `yield`).

The test exercises: (a) basic per-name sum, (b) exception propagation with accumulation preserved, (c) nested same-name re-entry summing both inner and outer elapsed.

In [ ]:
import time
import contextlib

@contextlib.contextmanager
def ex2_stage(name, acc):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - t0
        acc[name] = acc.get(name, 0.0) + elapsed


<details><summary>Solution</summary>

```python
import time
import contextlib

@contextlib.contextmanager
def ex2_stage(name, acc):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - t0
        acc[name] = acc.get(name, 0.0) + elapsed
```

**`try/finally` is the load-bearing structural choice.** A try/except would let you handle the exception inside the CM, but you'd have to re-raise to preserve user semantics. `finally` guarantees the accumulation regardless of how the block exits — normal return, exception, or even `return` inside a function wrapping the `with`.

**`acc.get(name, 0.0)` over `if name in acc`.** One line. Default handling baked in. No race between the existence check and the assignment (irrelevant single-threaded, but a habit worth keeping).

**Nested same-name accumulates by design.** Outer block's elapsed includes the inner block's elapsed (because perf_counter ticks the whole time). Inner block also adds its own elapsed. Total = outer + inner. For non-overlapping nested stages this is the correct sum; for overlapping you'd want a different design.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()